# Deploy Claude Code on GCP via Vertex AI

This notebook walks you through deploying the full reference
architecture **from within Colab or Vertex Workbench**. You won't
need a local terminal. Each cell explains what it's about to do.

What you'll end up with:

- An **LLM gateway** (Cloud Run) that routes Claude Code traffic
  through Vertex AI.
- An **MCP gateway** (Cloud Run) hosting shared tools.
- A **dev portal** (Cloud Run) with per-OS setup instructions.
- *(optional)* A **shared dev VM** accessible via IAP TCP tunnel.
- *(optional)* **Observability**: a BigQuery log sink and Looker
  Studio template.

Prereqs (assumed true if you're running this in Colab or Vertex
Workbench with your Google account):

- You're authenticated to Google.
- You have `Owner` or `Editor` on the target project.
- Billing is enabled on the project.

> Run the cells top-to-bottom. Any step that mutates GCP prints
> a summary and waits for you to acknowledge before continuing.


## 1. Choose deployment settings

Edit the variables below. These mirror `config.yaml` in the repo —
see `config.example.yaml` for what each one means.


In [ ]:
# Edit these for your deployment, then run the cell.
PROJECT_ID        = 'my-gcp-project'          # REQUIRED
REGION            = 'global'                  # Vertex region
FALLBACK_REGION   = 'us-central1'             # GCE/Cloud Run region when REGION is 'global'

ENABLE_LLM        = True
ENABLE_MCP        = True
ENABLE_PORTAL     = True
ENABLE_DEV_VM     = False                     # costs money when True
ENABLE_OBS        = True

# Comma-separated list, IAM member format.
ALLOWED_PRINCIPALS = 'user:[email protected]'

# Pinned model IDs.
MODEL_OPUS   = 'claude-opus-4-6'
MODEL_SONNET = 'claude-sonnet-4-6'
MODEL_HAIKU  = 'claude-haiku-4-5@20251001'

print('Settings captured. Running target:', PROJECT_ID, 'in', REGION)


## 2. Authenticate and select the project

In Colab the helper below triggers a Google sign-in popup. In
Vertex Workbench you're already authenticated and this is a
no-op.


In [ ]:
import subprocess, sys

def sh(cmd: str, check: bool = True) -> str:
    """Run a shell command, stream output, and return stdout."""
    print(f'$ {cmd}')
    proc = subprocess.run(cmd, shell=True, check=check, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout)
    if proc.stderr.strip():
        print(proc.stderr, file=sys.stderr)
    return proc.stdout

# Colab-only auth helper; harmless if unavailable.
try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    print('Colab auth ok')
except Exception as e:
    print('Not in Colab, skipping colab auth:', e)

sh(f'gcloud config set project {PROJECT_ID}')
sh('gcloud auth list')


## 3. Enable the required APIs

This is idempotent — if the APIs are already enabled, nothing
changes.


In [ ]:
REQUIRED_APIS = [
    'aiplatform.googleapis.com', 'run.googleapis.com',
    'compute.googleapis.com',    'iap.googleapis.com',
    'artifactregistry.googleapis.com', 'cloudbuild.googleapis.com',
    'logging.googleapis.com',    'monitoring.googleapis.com',
    'iamcredentials.googleapis.com', 'secretmanager.googleapis.com',
    'serviceusage.googleapis.com', 'bigquery.googleapis.com',
]
sh(f'gcloud services enable {" ".join(REQUIRED_APIS)}')


## 4. Clone the repo

We need the source trees for the LLM gateway, MCP gateway, and
dev portal so we can build their container images.

⚠️ Edit `REPO_URL` if you've forked this project.


In [ ]:
import os
REPO_URL = 'https://github.com/PTA-Co-innovation-Team/Anthropic-Google-Co-Innovation.git'
REPO_SUBDIR = '05-solution-accelerators/claude-code-vertex-gcp'
REPO_DIR = '/content/Anthropic-Google-Co-Innovation' if os.path.isdir('/content') else '/tmp/Anthropic-Google-Co-Innovation'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull --ff-only')
# Point to the solution accelerator subdirectory for scripts
PROJECT_ROOT = os.path.join(REPO_DIR, REPO_SUBDIR)
print('Source tree at', PROJECT_ROOT)

## 5. Export config to the shell

The component scripts read from the environment. We set the same
variables `deploy.sh` would set.


In [ ]:
os.environ['PROJECT_ID']       = PROJECT_ID
os.environ['REGION']           = REGION
os.environ['FALLBACK_REGION']  = FALLBACK_REGION
os.environ['PRINCIPALS']       = ALLOWED_PRINCIPALS
print('Env exported')


## 6. Confirm before creating resources

This is your last chance to back out. Everything after this cell
may incur GCP charges.


In [ ]:
reply = input(f'Proceed and create resources in project {PROJECT_ID}? [yes/NO]: ')
assert reply.strip().lower() in ('y', 'yes'), 'Aborted.'
print('Proceeding.')


## 7. Deploy the LLM gateway

Runs `scripts/deploy-llm-gateway.sh` from the cloned repo. Takes
~3–5 minutes (Cloud Build).


In [ ]:
if ENABLE_LLM:
    sh(f'bash {PROJECT_ROOT}/scripts/deploy-llm-gateway.sh')
else:
    print('LLM gateway disabled, skipping.')

## 8. Deploy the MCP gateway


In [ ]:
if ENABLE_MCP:
    sh(f'bash {PROJECT_ROOT}/scripts/deploy-mcp-gateway.sh')
else:
    print('MCP gateway disabled, skipping.')

## 9. Deploy the dev portal

Reads the gateway URLs created above, substitutes them into the
static site, builds, and deploys.


In [ ]:
if ENABLE_PORTAL:
    sh(f'bash {PROJECT_ROOT}/scripts/deploy-dev-portal.sh')
else:
    print('Dev portal disabled, skipping.')

## 10. (Optional) Deploy the dev VM

Skipped by default. Turn on by setting `ENABLE_DEV_VM = True` in
the settings cell at the top.


In [ ]:
if ENABLE_DEV_VM:
    sh(f'bash {PROJECT_ROOT}/scripts/deploy-dev-vm.sh')
else:
    print('Dev VM disabled, skipping.')

## 11. Print the deployment summary

URLs + SSH command you'll need on the developer side.


In [ ]:
def get_url(svc):
    try:
        out = subprocess.check_output(
            ['gcloud', 'run', 'services', 'describe', svc,
             '--project', PROJECT_ID, '--region', FALLBACK_REGION,
             '--format=value(status.url)'],
            text=True,
        ).strip()
        return out
    except subprocess.CalledProcessError:
        return '(not deployed)'

print('=' * 60)
print('Deployment summary')
print('=' * 60)
print(f'Project:      {PROJECT_ID}')
print(f'Vertex region:{REGION}')
print(f'LLM gateway:  {get_url("llm-gateway")}')
print(f'MCP gateway:  {get_url("mcp-gateway")}')
print(f'Dev portal:   {get_url("dev-portal")}')
if ENABLE_DEV_VM:
    print(f'Dev VM SSH:   gcloud compute ssh --tunnel-through-iap '
          f'--project={PROJECT_ID} --zone={FALLBACK_REGION}-a claude-code-dev-shared')


## 12. (Optional) Clean up — run only when you're done

This tears everything down. Re-running cells above will recreate
the services, but any BigQuery log history will already be gone
if you also drop the dataset.


In [ ]:
# Uncomment to run teardown.
# import subprocess
# subprocess.run(['bash', f'{PROJECT_ROOT}/scripts/teardown.sh'], check=True)
print('Teardown cell is commented out by default. Uncomment to run.')

## Next steps

On each developer's laptop, run:

```bash
git clone https://github.com/PTA-Co-innovation-Team/Anthropic-Google-Co-Innovation.git
cd Anthropic-Google-Co-Innovation/05-solution-accelerators/claude-code-vertex-gcp/scripts
./developer-setup.sh
```

The script writes `~/.claude/settings.json` with the gateway URL
and runs a smoke test.

See the repo's `README.md` and `TROUBLESHOOTING.md` for more.